## 2 卷积和池化层

### 2.1 理论计算题

输入图像大小 $3\times32\times32$（通道 × 高 × 宽），卷积层有 16 个大小为 $3\times5\times5$ 的卷积核，Padding = 2，Stride = 2。

**(1) 输出特征图尺寸**

输出空间尺寸公式：
$$O = \left\lfloor \frac{I + 2P - K}{S} \right\rfloor + 1$$

代入 $I=32,\ P=2,\ K=5,\ S=2$：
$$O = \left\lfloor \frac{32 + 2\times2 - 5}{2} \right\rfloor + 1 = \left\lfloor \frac{31}{2} \right\rfloor + 1 = 15 + 1 = 16$$

输出通道数等于卷积核个数 = 16。

$$\boxed{\text{输出特征图尺寸 } = 16 \times 16 \times 16\ (\text{通道} \times \text{高} \times \text{宽})}$$

**(2) 单个输出像素需要的点乘（乘法）次数**

单个输出通道上的一个像素值，是用一个卷积核 $3\times5\times5$ 与输入对应区域做点乘求和得到的，因此乘法次数等于卷积核中的元素个数：

$$3 \times 5 \times 5 = \boxed{75 \text{ 次乘法}}$$

### 2.2 编程题：手动实现二维最大池化

不使用 `torch.nn.MaxPool2d`，仅用 NumPy 实现支持 stride 和 padding 的二维最大池化前向传播。

> 注意：最大池化的填充应使用 $-\infty$ 填充，保证填充值不会被错误地选为最大值。

In [1]:
import numpy as np


def max_pool2d_forward(x, kernel_size, stride=None, padding=0):
    """二维最大池化前向传播（手动实现）。

    参数:
        x: 输入张量, 形状 (N, C, H, W)
        kernel_size: 池化窗口大小 (int)
        stride: 步幅, 默认为 kernel_size
        padding: 填充大小, 用 -inf 填充
    返回:
        out: 池化后的张量, 形状 (N, C, H_out, W_out)
    """
    if stride is None:
        stride = kernel_size

    N, C, H, W = x.shape
    # 最大池化用 -inf 填充, 避免填充值影响最大值
    x_pad = np.pad(
        x,
        ((0, 0), (0, 0), (padding, padding), (padding, padding)),
        mode="constant",
        constant_values=-np.inf,
    )
    H_pad, W_pad = x_pad.shape[2], x_pad.shape[3]

    H_out = (H_pad - kernel_size) // stride + 1
    W_out = (W_pad - kernel_size) // stride + 1

    out = np.zeros((N, C, H_out, W_out), dtype=x.dtype)
    for i in range(H_out):
        for j in range(W_out):
            h_start = i * stride
            w_start = j * stride
            window = x_pad[:, :, h_start:h_start + kernel_size,
                              w_start:w_start + kernel_size]
            out[:, :, i, j] = window.max(axis=(2, 3))
    return out


# ===== 测试 =====
x = np.arange(16, dtype=np.float32).reshape(1, 1, 4, 4)
print("输入 (1x1x4x4):")
print(x[0, 0])

out1 = max_pool2d_forward(x, kernel_size=2, stride=2, padding=0)
print("\n最大池化 kernel=2, stride=2, padding=0  ->  形状", out1.shape)
print(out1[0, 0])

out2 = max_pool2d_forward(x, kernel_size=3, stride=1, padding=1)
print("\n最大池化 kernel=3, stride=1, padding=1  ->  形状", out2.shape)
print(out2[0, 0])

输入 (1x1x4x4):
[[ 0.  1.  2.  3.]
 [ 4.  5.  6.  7.]
 [ 8.  9. 10. 11.]
 [12. 13. 14. 15.]]

最大池化 kernel=2, stride=2, padding=0  ->  形状 (1, 1, 2, 2)
[[ 5.  7.]
 [13. 15.]]

最大池化 kernel=3, stride=1, padding=1  ->  形状 (1, 1, 4, 4)
[[ 5.  6.  7.  7.]
 [ 9. 10. 11. 11.]
 [13. 14. 15. 15.]
 [13. 14. 15. 15.]]


## 3 LeNet, AlexNet, VGG 和 NiN

### 3.1 理论计算题

输入与输出特征图通道数均为 $C$，卷积核不带偏置。

**(1) 一个 $5\times5$ 卷积层的参数量**

参数量 = 输出通道 × 输入通道 × 卷积核高 × 卷积核宽：
$$C \times C \times 5 \times 5 = \boxed{25C^2}$$

**(2) 两个串联的 $3\times3$ 卷积层的总参数量**

每个 $3\times3$ 卷积层（输入输出通道都为 $C$）的参数量：
$$C \times C \times 3 \times 3 = 9C^2$$

两层串联：
$$2 \times 9C^2 = \boxed{18C^2}$$

**结论**：$18C^2 < 25C^2$。两个串联的 $3\times3$ 卷积在拥有与单个 $5\times5$ 卷积相同感受野（$5\times5$）的同时，参数更少，且因为多了一层非线性激活而表达能力更强，这正是 VGG 用小卷积核级联替代大卷积核的原因。

### 3.2 编程题：定义 NiN 块

使用 `torch.nn.Sequential` 定义一个标准 NiN 块：一个普通卷积层 + 两个 $1\times1$ 卷积层，每个卷积层后紧跟 ReLU。

In [2]:
import torch
from torch import nn


def nin_block(in_channels, out_channels, kernel_size, stride, padding):
    """构造一个 NiN 块。

    由一个普通卷积层 + 两个 1x1 卷积层级联组成,
    每个卷积层后都紧跟一个 ReLU 激活层。
    """
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding),
        nn.ReLU(),
        nn.Conv2d(out_channels, out_channels, kernel_size=1),
        nn.ReLU(),
        nn.Conv2d(out_channels, out_channels, kernel_size=1),
        nn.ReLU(),
    )


# ===== 测试 =====
block = nin_block(in_channels=3, out_channels=96,
                  kernel_size=11, stride=4, padding=0)
print(block)

X = torch.rand(size=(1, 3, 224, 224))
Y = block(X)
print("\n输入形状:", tuple(X.shape))
print("输出形状:", tuple(Y.shape))

Sequential(
  (0): Conv2d(3, 96, kernel_size=(11, 11), stride=(4, 4))
  (1): ReLU()
  (2): Conv2d(96, 96, kernel_size=(1, 1), stride=(1, 1))
  (3): ReLU()
  (4): Conv2d(96, 96, kernel_size=(1, 1), stride=(1, 1))
  (5): ReLU()
)

输入形状: (1, 3, 224, 224)
输出形状: (1, 96, 54, 54)


## 4 Inception, 批量归一化和残差网络

### 4.1 理论计算题：Batch Normalization

4 个样本在某通道某空间位置的值：$x_1=2,\ x_2=4,\ x_3=6,\ x_4=8$，参数 $\gamma=2,\ \beta=1,\ \epsilon=0$。

**第一步：均值**
$$\mu = \frac{2+4+6+8}{4} = 5$$

**第二步：方差**（批量归一化使用有偏方差，除以 $N$）
$$\sigma^2 = \frac{(2-5)^2+(4-5)^2+(6-5)^2+(8-5)^2}{4} = \frac{9+1+1+9}{4} = 5$$
$$\sigma = \sqrt{5} \approx 2.2361$$

**第三步：标准化** $\hat{x}_i = \dfrac{x_i - \mu}{\sqrt{\sigma^2+\epsilon}}$
$$\hat{x}_1 = \frac{-3}{\sqrt5},\ \hat{x}_2 = \frac{-1}{\sqrt5},\ \hat{x}_3 = \frac{1}{\sqrt5},\ \hat{x}_4 = \frac{3}{\sqrt5}$$

**第四步：缩放平移** $y_i = \gamma\hat{x}_i + \beta = 2\hat{x}_i + 1$

$$\boxed{y_1 \approx -1.6833,\quad y_2 \approx 0.1056,\quad y_3 \approx 1.8944,\quad y_4 \approx 3.6833}$$

下面用代码验证：

In [3]:
import numpy as np

x = np.array([2.0, 4.0, 6.0, 8.0])
gamma, beta, eps = 2.0, 1.0, 0.0

mu = x.mean()
var = x.var()            # 有偏方差(除以 N), 与 BN 训练时一致
x_hat = (x - mu) / np.sqrt(var + eps)
y = gamma * x_hat + beta

print("均值 mu =", mu)
print("方差 var =", var, " 标准差 =", round(float(np.sqrt(var)), 4))
print("标准化 x_hat =", np.round(x_hat, 4))
print("最终输出 y   =", np.round(y, 4))

均值 mu = 5.0
方差 var = 5.0  标准差 = 2.2361
标准化 x_hat = [-1.3416 -0.4472  0.4472  1.3416]
最终输出 y   = [-1.6833  0.1056  1.8944  3.6833]


### 4.2 编程题：自定义残差块 Residual

包含两个相同输出通道数的 $3\times3$ 卷积层，每个卷积后接批量归一化层。当 `use_1x1conv=True` 时，对输入用 $1\times1$ 卷积调整通道数与形状，使其能与第二层输出按元素相加 $f(x)+x$。

In [4]:
import torch
from torch import nn
from torch.nn import functional as F


class Residual(nn.Module):
    """残差块 (ResNet Block)。"""

    def __init__(self, in_channels, out_channels,
                 use_1x1conv=False, strides=1):
        super().__init__()
        # 两个 3x3 卷积层, 第一个可通过 strides 改变空间尺寸
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3,
                               padding=1, stride=strides)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3,
                               padding=1)
        # 每个卷积层后跟一个批量归一化层
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.bn2 = nn.BatchNorm2d(out_channels)
        # 是否用 1x1 卷积调整输入的通道数和形状
        if use_1x1conv:
            self.conv3 = nn.Conv2d(in_channels, out_channels, kernel_size=1,
                                   stride=strides)
        else:
            self.conv3 = None

    def forward(self, X):
        Y = F.relu(self.bn1(self.conv1(X)))
        Y = self.bn2(self.conv2(Y))
        if self.conv3 is not None:
            X = self.conv3(X)
        Y += X                  # 残差连接 f(x) + x
        return F.relu(Y)


# ===== 测试 =====
# 情况一: 输入输出通道一致, 形状不变
blk1 = Residual(3, 3)
X = torch.rand(4, 3, 6, 6)
print("情况一 (use_1x1conv=False): 输入", tuple(X.shape),
      "-> 输出", tuple(blk1(X).shape))

# 情况二: 改变通道数并减半空间尺寸, 需要 1x1 卷积
blk2 = Residual(3, 6, use_1x1conv=True, strides=2)
print("情况二 (use_1x1conv=True, strides=2): 输入", tuple(X.shape),
      "-> 输出", tuple(blk2(X).shape))

情况一 (use_1x1conv=False): 输入 (4, 3, 6, 6) -> 输出 (4, 3, 6, 6)
情况二 (use_1x1conv=True, strides=2): 输入 (4, 3, 6, 6) -> 输出 (4, 6, 3, 3)


## 5 图像增广，微调和样式迁移

### 5.1 理论计算题：微调

**(1) 为什么底层特征提取层用较小学习率（甚至冻结），而顶层输出层用较大学习率？**

- 在大型源数据集（如 ImageNet）上预训练得到的**底层特征提取层**已经学到了通用、可迁移的低层特征（如边缘、纹理、颜色、基本形状），这些特征对新任务通常依然有效。用较小学习率（甚至冻结）可以**保留这些宝贵的预训练知识**，避免在小目标数据集上被噪声梯度破坏，同时减少计算量、加快收敛。
- 而**顶层输出层**是针对新任务**随机重新初始化**的，它还没有学到任何东西，需要从头快速学习以适配新数据集的类别和分布，因此设置较大的学习率让它更快地拟合目标任务。
- 总之：这种差异化学习率实现了"**保留通用知识 + 快速适配新任务**"的平衡，既防止灾难性遗忘，又加速对新任务的学习。

**(2) 如果目标数据集非常小，且与源数据集非常相似，应采取什么微调策略以防止过拟合？**

- **冻结绝大部分（甚至全部）底层特征提取层**，只训练（重新初始化的）顶层输出层（即把预训练网络当作固定的特征提取器，linear probing）。因为数据集小，可训练参数越多越容易过拟合；又因为两数据集很相似，预训练特征几乎可以直接复用，无需微调底层。
- 此外可配合：使用较小的学习率、较强的**正则化**（权重衰减、Dropout）、以及**数据增广**（见 5.2）来进一步扩充数据多样性、抑制过拟合。

### 5.2 编程题：组合图像增广管道

利用 `torchvision.transforms` 创建组合增广管道：
1. 随机裁剪，面积比例 0.08~1.0，缩放到 $224\times224$；
2. 50% 概率水平翻转；
3. 随机改变亮度、对比度、饱和度，变化范围 0.5；
4. 转换为 PyTorch 张量。

In [5]:
import torchvision.transforms as transforms

augmentation_pipeline = transforms.Compose([
    # 1. 随机裁剪, 面积比例 0.08~1.0, 缩放到 224x224
    transforms.RandomResizedCrop(size=224, scale=(0.08, 1.0)),
    # 2. 50% 概率水平翻转
    transforms.RandomHorizontalFlip(p=0.5),
    # 3. 随机改变亮度/对比度/饱和度, 变化范围 0.5
    transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5),
    # 4. 转换为 PyTorch 张量
    transforms.ToTensor(),
])

print(augmentation_pipeline)

# ===== 测试: 用一张随机生成的 PIL 图像走一遍管道 =====
from PIL import Image
import numpy as np

dummy = Image.fromarray(
    (np.random.rand(300, 400, 3) * 255).astype("uint8"))
out = augmentation_pipeline(dummy)
print("\n增广后输出张量形状:", tuple(out.shape))   # 期望 (3, 224, 224)
print("张量取值范围: [%.3f, %.3f]" % (out.min().item(), out.max().item()))

Compose(
    RandomResizedCrop(size=(224, 224), scale=(0.08, 1.0), ratio=(0.75, 1.3333), interpolation=bilinear, antialias=True)
    RandomHorizontalFlip(p=0.5)
    ColorJitter(brightness=(0.5, 1.5), contrast=(0.5, 1.5), saturation=(0.5, 1.5), hue=None)
    ToTensor()
)

增广后输出张量形状: (3, 224, 224)
张量取值范围: [0.000, 0.741]


## 6 目标检测，计算机视觉训练技巧

### 6.1 理论计算题：IoU

真实框 $A=[10,10,50,50]$，预测框 $B=[30,30,70,70]$（格式：左上角 x, 左上角 y, 右下角 x, 右下角 y）。

**交集区域**（取两框左上角的较大值、右下角的较小值）：
$$x_{left}=\max(10,30)=30,\quad y_{top}=\max(10,30)=30$$
$$x_{right}=\min(50,70)=50,\quad y_{bottom}=\min(50,70)=50$$
$$\text{交集面积} = (50-30)\times(50-30) = 20\times20 = 400$$

**各框面积**：
$$S_A = (50-10)\times(50-10) = 1600,\qquad S_B = (70-30)\times(70-30) = 1600$$

**并集面积**：
$$\text{并集} = S_A + S_B - \text{交集} = 1600 + 1600 - 400 = 2800$$

**IoU**：
$$\text{IoU} = \frac{400}{2800} = \frac{1}{7} \approx \boxed{0.142857}$$

In [6]:
def compute_iou(box_a, box_b):
    """计算两个边界框的 IoU。框格式: [x1, y1, x2, y2]。"""
    # 交集矩形
    x_left = max(box_a[0], box_b[0])
    y_top = max(box_a[1], box_b[1])
    x_right = min(box_a[2], box_b[2])
    y_bottom = min(box_a[3], box_b[3])

    inter_w = max(0, x_right - x_left)
    inter_h = max(0, y_bottom - y_top)
    inter_area = inter_w * inter_h

    area_a = (box_a[2] - box_a[0]) * (box_a[3] - box_a[1])
    area_b = (box_b[2] - box_b[0]) * (box_b[3] - box_b[1])
    union_area = area_a + area_b - inter_area

    return inter_area / union_area if union_area > 0 else 0.0


A = [10, 10, 50, 50]
B = [30, 30, 70, 70]
iou = compute_iou(A, B)
print("交集面积 =", (50 - 30) * (50 - 30))
print("并集面积 =", 1600 + 1600 - 400)
print("IoU = %.6f  (= 1/7 = %.6f)" % (iou, 1 / 7))

交集面积 = 400
并集面积 = 2800
IoU = 0.142857  (= 1/7 = 0.142857)


### 6.2 编程题：标签平滑交叉熵损失

标签平滑：真实类别目标概率从 1 变为 $1-\epsilon$，其余 $K-1$ 个错误类别的概率从 0 变为 $\dfrac{\epsilon}{K-1}$。损失为软标签与 log-softmax 的负点积。

In [7]:
import torch
import torch.nn.functional as F


def label_smoothing_cross_entropy(logits, targets, eps=0.1):
    """计算标签平滑后的交叉熵损失。

    参数:
        logits:  模型输出, 形状 (N, K)  (未经过 softmax)
        targets: 真实类别索引, 形状 (N,)
        eps:     平滑因子
    返回:
        标量损失 (对 batch 取平均)
    """
    N, K = logits.shape
    log_probs = F.log_softmax(logits, dim=1)            # (N, K)

    # 构造平滑后的软标签
    true_prob = 1.0 - eps
    other_prob = eps / (K - 1)
    smooth_targets = torch.full_like(logits, other_prob)
    smooth_targets.scatter_(1, targets.unsqueeze(1), true_prob)

    # 交叉熵: -sum(软标签 * log_softmax), 再对 batch 取平均
    loss = -(smooth_targets * log_probs).sum(dim=1).mean()
    return loss


# ===== 测试 =====
logits = torch.tensor([[2.0, 0.5, 0.1],
                       [0.2, 1.5, 0.3]])
targets = torch.tensor([0, 1])

loss_smooth = label_smoothing_cross_entropy(logits, targets, eps=0.1)
loss_standard = F.cross_entropy(logits, targets)   # 标准交叉熵对照

print("标签平滑交叉熵损失 (eps=0.1):", round(loss_smooth.item(), 6))
print("标准交叉熵损失 (对照)      :", round(loss_standard.item(), 6))

标签平滑交叉熵损失 (eps=0.1): 0.532612
标准交叉熵损失 (对照)      : 0.385112
